In [2]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

# Veri setini yükle
CSV_FILENAME = "../datasets/student_dataset_10000_rows.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (10000, 8)


In [3]:
# ==========================================
# 1.1 İLK KEŞİF
# ==========================================

print("\n" + "=" * 50)
print("🔍 EKSİK VERİ ANALİZİ (Missing Values)")
print("=" * 50)

# Sadece eksik verisi olan sütunları ve oranlarını hesapla
missing_count = df.isnull().sum()
missing_percent = 100 * df.isnull().mean()

missing_df = pd.DataFrame(
    {"Eksik Sayısı": missing_count, "Oran (%)": missing_percent}
)
# Sadece eksik değeri olanları filtrele ve büyükten küçüğe sırala
missing_df = missing_df[missing_df["Eksik Sayısı"] > 0].sort_values(
    by="Eksik Sayısı", ascending=False
)

if missing_df.empty:
  print("✨ Harika! Veri setinde hiç eksik değer yok.")
else:
  print(missing_df.to_string())

print("=" * 50 + "\n")


🔍 EKSİK VERİ ANALİZİ (Missing Values)
✨ Harika! Veri setinde hiç eksik değer yok.



In [4]:
# ==========================================
# 1.2 KEŞİFSEL VERİ ANALİZİ
# ==========================================
# 1. Genel Yapı ve Tipler
print("--- Bilgiler ve Tipler ---")
print(df.info())

# 2. İstatistiksel Dağılım
print("\n--- İstatistiksel Özet ---")
display(df.describe())

# 3. Eksik Değer Kontrolü
print("\n--- Eksik Değerler ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# 4. İlk 5 Satır Göz Atma
print("\n--- İlk 5 Satır ---")
display(df.head())

--- Bilgiler ve Tipler ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   study_hours            10000 non-null  int64  
 1   attendance             10000 non-null  int64  
 2   sleep_hours            10000 non-null  int64  
 3   internet_usage         10000 non-null  int64  
 4   assignments_completed  10000 non-null  int64  
 5   previous_score         10000 non-null  int64  
 6   exam_score             10000 non-null  float64
 7   placement_status       10000 non-null  int64  
dtypes: float64(1), int64(7)
memory usage: 625.1 KB
None

--- İstatistiksel Özet ---


,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score,placement_status
count,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000
mean,5.989600,69.88460,6.498500,6.062600,9.988400,64.91100,86.704207,0.835600
std,3.163589,17.61653,1.709354,3.138163,6.034145,17.50302,15.058383,0.370657
min,1.000000,40.00000,4.000000,1.000000,0.000000,35.00000,26.670000,0.000000
25%,3.000000,55.00000,5.000000,3.000000,5.000000,50.00000,76.727500,1.000000
50%,6.000000,70.00000,6.500000,6.000000,10.000000,65.00000,92.120000,1.000000
75%,9.000000,85.00000,8.000000,9.000000,15.000000,80.00000,100.000000,1.000000
max,11.000000,100.00000,9.000000,11.000000,20.000000,95.00000,100.000000,1.000000



--- Eksik Değerler ---
Series([], dtype: int64)

--- İlk 5 Satır ---


,study_hours,attendance,sleep_hours,internet_usage,assignments_completed,previous_score,exam_score,placement_status
0,7,56,8,7,10,62,100.00,1
1,4,69,5,3,8,56,100.00,1
2,11,60,7,6,10,45,100.00,1
3,8,99,9,8,4,55,90.17,1
4,5,52,8,6,8,40,78.82,1


In [5]:
# ==========================================
# 1.3 VERİ TEMİZLEME VE ÖN İŞLEME
# ==========================================

X = df.drop(columns=["placement_status"])
y = df["placement_status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
# ==========================================
# 1.4 ÖZNİTELİK MÜHENDİSLİĞİ
# ==========================================

numeric_cols = [
    "study_hours",
    "attendance",
    "sleep_hours",
    "internet_usage",
    "assignments_completed",
    "previous_score",
    "exam_score",
]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_cols)],
    remainder="passthrough",
)

pipeline_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=10)),
        (
            "classifier",
            XGBClassifier(
                subsample=0.8,
                n_estimators=180,
                min_child_weight=3,
                max_depth=2,
                learning_rate=0.05,
                colsample_bytree=1.0,
                random_state=42,
            ),
        ),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 1.0000
Fold 2: 1.0000
Fold 3: 1.0000
Fold 4: 0.9995
Fold 5: 1.0000

Gerçek K-Fold Başarısı (Ortalama): 0.9999
Skor Sapması (Standart Sapma)    : 0.0002



In [8]:
# ==========================================
# ARA BÖLÜM: İDEAL K DEĞERİNİ BULMA
# ==========================================
for k_val in [10, 15, 20, 25, 30]:
    temp_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k=k_val)),
        ("classifier", XGBClassifier(subsample=0.8,
                n_estimators=180,
                min_child_weight=3,
                max_depth=2,
                learning_rate=0.05,
                colsample_bytree=1.0,
                random_state=42,)),
    ])

    scores = cross_val_score(temp_pipeline, X, y, cv=skf, scoring='accuracy')
    print(f"k = {k_val} için Ortalama K-Fold Başarısı: {np.mean(scores):.4f}")

k = 10 için Ortalama K-Fold Başarısı: 0.9999
k = 15 için Ortalama K-Fold Başarısı: 0.9999
k = 20 için Ortalama K-Fold Başarısı: 0.9998
k = 25 için Ortalama K-Fold Başarısı: 0.9998
k = 30 için Ortalama K-Fold Başarısı: 0.9998


In [10]:
# ==========================================
# 2. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))



Train Doğruluk Oranı: 1.0000
Model Doğruluk Oranı (Accuracy): 1.0000

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       329
           1       1.00      1.00      1.00      1671

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [11]:
# ==========================================
# 3. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
preprocessor = pipeline_model.named_steps["preprocessor"]
selector = pipeline_model.named_steps["feature_selection"]
classifier = pipeline_model.named_steps["classifier"]

all_feature_names = preprocessor.get_feature_names_out()
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask]

# RandomForest için coef_ yerine feature_importances_ kullanılır
importances = classifier.feature_importances_

feature_importance = pd.DataFrame(
    {
        "Özellik (Feature)": selected_features,
        "Önem Düzeyi (Importance)": importances,
    }
)

# --- GÖRSEL TEMİZLİK VE SIRALAMA BÖLÜMÜ ---
feature_importance["Özellik (Feature)"] = feature_importance[
    "Özellik (Feature)"
].str.replace(r"^(num__|cat__|text__|remainder__)", "", regex=True)

# Önem düzeyine göre azalan şekilde sıralayalım (mutlak değer almaya gerek yok, importances hep pozitiftir)
feature_importance = feature_importance.sort_values(
    by="Önem Düzeyi (Importance)", ascending=False
)

feature_importance["Önem Düzeyi (Importance)"] = feature_importance[
    "Önem Düzeyi (Importance)"
].round(4)
# ------------------------------------------

print("\n--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---")
print(feature_importance.to_string(index=False))

print(
    "En düşük tahmin edilen olasılık:",
    pipeline_model.predict_proba(X_test)[:, 1].min(),
)


--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---
                Özellik (Feature)  Önem Düzeyi (Importance)
                     exam_score^2                    0.8876
                       exam_score                    0.1124
            assignments_completed                    0.0000
                      study_hours                    0.0000
study_hours assignments_completed                    0.0000
                   previous_score                    0.0000
           study_hours exam_score                    0.0000
            attendance exam_score                    0.0000
 assignments_completed exam_score                    0.0000
        previous_score exam_score                    0.0000
En düşük tahmin edilen olasılık: 0.0027151534


In [13]:
# ==========================================
# 4. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/student_pipeline.pkl")
joblib.dump(list(X_train.columns), "../backend/models/model_columns_xgboost.pkl")

print(
    "Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne"
    " kaydedildi!"
)

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!
